<a href="https://colab.research.google.com/github/GigaGoriashvili/two-headed-mlp/blob/main/mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Task Learning with a Two-Headed MLP
#### Giga Goriashvili

## Part 1 - Preprocessing the Data

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

In [ ]:
try:
    df = pd.read_csv('student-por.csv', sep=';')
except FileNotFoundError:
    print("Error: 'student-por.csv' file not found.")

print("Data loaded successfully:")
print(df.head())

Data loaded successfully:
  school sex  age address famsize Pstatus  Medu  Fedu     Mjob      Fjob  ...  \
0     GP   F   18       U     GT3       A     4     4  at_home   teacher  ...   
1     GP   F   17       U     GT3       T     1     1  at_home     other  ...   
2     GP   F   15       U     LE3       T     1     1  at_home     other  ...   
3     GP   F   15       U     GT3       T     4     2   health  services  ...   
4     GP   F   16       U     GT3       T     3     3    other     other  ...   

  famrel freetime  goout  Dalc  Walc health absences  G1  G2  G3  
0      4        3      4     1     1      3        4   0  11  11  
1      5        3      3     1     1      3        2   9  11  11  
2      4        3      2     2     3      3        6  12  13  12  
3      3        2      2     1     1      5        0  14  14  14  
4      4        3      2     1     2      5        0  11  13  13  

[5 rows x 33 columns]


### Feature Selection

In [ ]:
y_grade = df['G3'].values.astype(np.float32)

y_romantic = df['romantic'].map({'no': 0, 'yes': 1}).values.astype(np.int64)

X = df.drop(columns=['G3', 'romantic'])

print(f"\n_y_grade sample: {y_grade[:5]}")
print(f"y_romantic sample: {y_romantic[:5]}")


_y_grade sample: [11. 11. 12. 14. 13.]
y_romantic sample: [0 0 0 1 0]


In [ ]:
# Numerical features
numeric_features = [
    'age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures',
    'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences',
    'G1', 'G2'
]

# Categorical features
categorical_features = [
    'school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
    'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities',
    'nursery', 'higher', 'internet'
]

# Ensuring all columns are included
assert len(numeric_features) + len(categorical_features) == len(X.columns)

### Preprocessing Pipelines

In [ ]:
# Pipeline for numerical features:
# Using Standard Scaler
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Pipeline for categorical features:
# Using OneHotEncoder
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # 'handle_unknown' for rare cases
])

# Combining these pipelines using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

### Data Splitting

In [ ]:
# First split: 80% train+val, 20% test
X_train_val, X_test, y_grade_train_val, y_grade_test, y_romantic_train_val, y_romantic_test = train_test_split(
    X, y_grade, y_romantic, test_size=0.2, random_state=42
)

# Second split: Split train+val (from the 80%) -> 80% train, 20% validation
# This will eventually give approximately 64% train, 16% val, 20% test
X_train, X_val, y_grade_train, y_grade_val, y_romantic_train, y_romantic_val = train_test_split(
    X_train_val, y_grade_train_val, y_romantic_train_val, test_size=0.2, random_state=42 # 0.2 of 80% is 16%
)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

Training set size: 415
Validation set size: 104
Test set size: 130


### Applying Preprocessing

In [ ]:
# Fitting the preprocessor only on the training data
preprocessor.fit(X_train)

# Applying the transformations
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# The number of columns will increase due to OneHotEncoding
print(f"\nNumber of features before processing: {X_train.shape[1]}")
print(f"Number of features after processing: {X_train_processed.shape[1]}")


Number of features before processing: 31
Number of features after processing: 56


### Custom PyTorch Dataset and DataLoaders

In [ ]:
class StudentDataset(Dataset):
    def __init__(self, X, y_grade, y_romantic):
        # Converting everything to PyTorch Tensors
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_grade = torch.tensor(y_grade, dtype=torch.float32)
        self.y_romantic = torch.tensor(y_romantic, dtype=torch.int64)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # According to the task requirement, we return 3 values
        return (
            self.X[idx],
            self.y_grade[idx],
            self.y_romantic[idx]
        )

# Creating Datasets for all three sets
train_dataset = StudentDataset(X_train_processed, y_grade_train, y_romantic_train)
val_dataset = StudentDataset(X_val_processed, y_grade_val, y_romantic_val)
test_dataset = StudentDataset(X_test_processed, y_grade_test, y_romantic_test)

# Checking one sample
x_sample, grade_sample, romantic_sample = train_dataset[0]
print(f"\nSample X shape: {x_sample.shape}")
print(f"Sample Grade: {grade_sample}")
print(f"Sample Romantic: {romantic_sample}")


Sample X shape: torch.Size([56])
Sample Grade: 12.0
Sample Romantic: 1


In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True)

val_loader = DataLoader(dataset=val_dataset,
                        batch_size=BATCH_SIZE,
                        shuffle=False)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=BATCH_SIZE,
                         shuffle=False)

print(f"\nCreated {len(train_loader)} batches in train_loader.")


Created 13 batches in train_loader.


## Part 2 - Building the Multi-Head Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### Creating Multi-Head Model Class

In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(self, input_features):
        super(MultiTaskModel, self).__init__()

        # --- 1. Shared Body ---
        # Takes 56 features and creates a 64-dimensional "profile"
        self.shared_body = nn.Sequential(
            # Layer 1: 56 -> 64
            nn.Linear(input_features, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),

            # Layer 2: 64 -> 128
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.4),

            # Layer 3: 128 -> 64 (Final profile)
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64)
        )

        # --- 2. Head 1: Grade Prediction (Regression) ---
        # Takes the 64-dimensional profile and returns 1 number
        self.head_grade = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        # --- 3. Head 2: Romantic Status (Classification) ---
        # Takes the 64-dimensional profile and returns 2 numbers (logits)
        self.head_romantic = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        # Data passes through the "body" to create the "profile"
        shared_features = self.shared_body(x)

        # The "profile" is fed to both "heads"
        output_grade = self.head_grade(shared_features)
        output_romantic = self.head_romantic(shared_features)

        # Return both results
        return output_grade, output_romantic

## Part 3 - The Custom Training Loop

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

### Defining Parameters and Loss Functions

In [ ]:
# --- Preparing Parameters ---
INPUT_FEATURES = 56
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiTaskModel(input_features=INPUT_FEATURES).to(device)


# Loss Function for 'Grade' Regression
loss_fn_grade = nn.MSELoss()
# Loss Function for 'Romantic' Classification
loss_fn_class = nn.CrossEntropyLoss()


# Optimizer Adam
optimizer = optim.Adam(model.parameters(), lr=0.001)


# 4. Hyperparameters
EPOCHS = 50
LEARNING_RATE = 0.001

# alpha = 0.8 means: "80% focus on Grade, 20% on Romantic"
ALPHA = 0.4

### Defining Training and Validating Functions

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn_grade, loss_fn_class, alpha, device):
    """
    Runs one training epoch for the model.
    """
    model.train() # Sets the model to training mode

    total_loss_sum = 0
    grade_loss_sum = 0
    romantic_loss_sum = 0

    for (x_batch, y_grade_batch, y_romantic_batch) in loader:
        # Data move to GPU
        x_batch = x_batch.to(device)
        y_grade_batch = y_grade_batch.to(device)
        y_romantic_batch = y_romantic_batch.to(device)

        # Forward pass through the model
        pred_grade, pred_romantic = model(x_batch)

        # Calculating losses separately
        loss_g = loss_fn_grade(pred_grade.squeeze(1), y_grade_batch)

        loss_r = loss_fn_class(pred_romantic, y_romantic_batch)

        # Combines losses
        total_loss = (alpha * loss_g) + ((1.0 - alpha) * loss_r)

        # Backpropagation
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        # Accumulate losses to calculate the average
        total_loss_sum += total_loss.item()
        grade_loss_sum += loss_g.item()
        romantic_loss_sum += loss_r.item()

    # Returns the average losses for this epoch
    return (total_loss_sum / len(loader),
            grade_loss_sum / len(loader),
            romantic_loss_sum / len(loader))

In [ ]:
def validate_epoch(model, loader, loss_fn_grade, loss_fn_class, alpha, device):
    """
    Runs one validation epoch for the model.
    """
    model.eval() # Sets the model to evaluation mode

    total_loss_sum = 0
    grade_loss_sum = 0
    romantic_loss_sum = 0

    with torch.no_grad():
        for (x_batch, y_grade_batch, y_romantic_batch) in loader:
            # Data Move to GPU/CPU
            x_batch = x_batch.to(device)
            y_grade_batch = y_grade_batch.to(device)
            y_romantic_batch = y_romantic_batch.to(device)

            # Model forward pass
            pred_grade, pred_romantic = model(x_batch)

            # Calculates losses
            loss_g = loss_fn_grade(pred_grade.squeeze(1), y_grade_batch)
            loss_r = loss_fn_class(pred_romantic, y_romantic_batch)

            # Combines losses
            total_loss = (alpha * loss_g) + ((1.0 - alpha) * loss_r)

            # Accumulates losses
            total_loss_sum += total_loss.item()
            grade_loss_sum += loss_g.item()
            romantic_loss_sum += loss_r.item()

    # Returns the average losses for this epoch
    return (total_loss_sum / len(loader),
            grade_loss_sum / len(loader),
            romantic_loss_sum / len(loader))

### Training Process

In [ ]:
# Starting Training

print(f"Starting Training: ALPHA={ALPHA}, EPOCHS={EPOCHS}, DEVICE={device}")

# Lists to store results (for plotting)
history = {
    'train_total': [], 'val_total': [],
    'train_grade': [], 'val_grade': [],
    'train_romantic': [], 'val_romantic': []
}

for epoch in range(EPOCHS):
    # Training Stage
    train_total, train_grade, train_romantic = train_epoch(
        model, train_loader, optimizer, loss_fn_grade, loss_fn_class, ALPHA, device
    )

    # Validation Stage
    val_total, val_grade, val_romantic = validate_epoch(
        model, val_loader, loss_fn_grade, loss_fn_class, ALPHA, device
    )

    # Stores History
    history['train_total'].append(train_total)
    history['val_total'].append(val_total)
    history['train_grade'].append(train_grade)
    history['val_grade'].append(val_grade)
    history['train_romantic'].append(train_romantic)
    history['val_romantic'].append(val_romantic)

    # --- Print Results ---
    print(f"Epoch {epoch+1:02}/{EPOCHS}")
    print(f"  Train: Total Loss: {train_total:.4f} | Grade Loss: {train_grade:.4f} | Romantic Loss: {train_romantic:.4f}")
    print(f"  Valid: Total Loss: {val_total:.4f} | Grade Loss: {val_grade:.4f} | Romantic Loss: {val_romantic:.4f}")

print("--- Training Finished ---")

Starting Training: ALPHA=0.4, EPOCHS=50, DEVICE=cuda
Epoch 01/50
  Train: Total Loss: 60.0943 | Grade Loss: 149.1801 | Romantic Loss: 0.7038
  Valid: Total Loss: 60.0132 | Grade Loss: 149.0301 | Romantic Loss: 0.6685
Epoch 02/50
  Train: Total Loss: 56.6264 | Grade Loss: 140.5417 | Romantic Loss: 0.6829
  Valid: Total Loss: 56.5204 | Grade Loss: 140.3183 | Romantic Loss: 0.6551
Epoch 03/50
  Train: Total Loss: 51.3787 | Grade Loss: 127.4658 | Romantic Loss: 0.6539
  Valid: Total Loss: 48.0803 | Grade Loss: 119.2469 | Romantic Loss: 0.6359
Epoch 04/50
  Train: Total Loss: 44.3807 | Grade Loss: 109.9673 | Romantic Loss: 0.6563
  Valid: Total Loss: 37.0689 | Grade Loss: 91.7462 | Romantic Loss: 0.6174
Epoch 05/50
  Train: Total Loss: 35.3761 | Grade Loss: 87.4616 | Romantic Loss: 0.6524
  Valid: Total Loss: 26.8233 | Grade Loss: 66.1530 | Romantic Loss: 0.6035
Epoch 06/50
  Train: Total Loss: 25.9041 | Grade Loss: 63.7823 | Romantic Loss: 0.6520
  Valid: Total Loss: 17.9491 | Grade Loss: 

In [ ]:
# Saves the *final* model
torch.save(model.state_dict(), 'my_model_weights.pth')
print("Model Weights are successfully stored in the file: 'my_model_weights.pth'")

Model Weights are successfully stored in the file: 'my_model_weights.pth'


## Model Evaluation

In [ ]:
import torch
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score
import numpy as np

In [ ]:
def evaluate_model(model, loader, device):
    """
    Evaluates the model on the test data (test_loader)
    and prints MAE, Accuracy, and F1-Score.
    """
    model.eval() # Sets the model to evaluation mode

    # Lists to collect all predictions and targets
    all_grade_targets = []
    all_grade_preds = []
    all_romantic_targets = []
    all_romantic_preds = []

    with torch.no_grad():
        for (x_batch, y_grade_batch, y_romantic_batch) in loader:
            # Moves data to GPU/CPU
            x_batch = x_batch.to(device)
            y_grade_batch = y_grade_batch.to(device)
            y_romantic_batch = y_romantic_batch.to(device)

            # Model forward pass
            pred_grade, pred_romantic = model(x_batch)

            # Stores Regression Results
            all_grade_targets.append(y_grade_batch.cpu())
            all_grade_preds.append(pred_grade.squeeze(1).cpu())

            # Stores Classification Results
            predicted_classes = torch.argmax(pred_romantic, dim=1)

            all_romantic_targets.append(y_romantic_batch.cpu())
            all_romantic_preds.append(predicted_classes.cpu())

    # Regression
    targets_g = torch.cat(all_grade_targets).numpy()
    preds_g = torch.cat(all_grade_preds).numpy()

    # Classification
    targets_r = torch.cat(all_romantic_targets).numpy()
    preds_r = torch.cat(all_romantic_preds).numpy()

    # Calculate Metrics

    # Grade Prediction (Regression)
    mae = mean_absolute_error(targets_g, preds_g)

    # Romantic Status (Classification)
    accuracy = accuracy_score(targets_r, preds_r)

    # F1-Score for the 'yes' class
    f1 = f1_score(targets_r, preds_r, pos_label=1)


    # Prints Final Results
    print("\n--- 📊 Final Testing Results (Test Set Performance) ---")
    print(f"\n📋 Task 1: Grade Prediction (Regression)")
    print(f"  Mean Absolute Error (MAE): {mae:.4f}")
    print(f"  (On average, the model is off by {mae:.4f} points)")

    print(f"\n📋 Task 2: Romantic Status (Classification)")
    print(f"  Accuracy: {accuracy * 100:.2f}%")
    print(f"  F1-Score (for 'yes' class): {f1:.4f}")
    print("-------------------------------------------------------------")

In [ ]:
evaluate_model(model, test_loader, device)


--- 📊 Final Testing Results (Test Set Performance) ---

📋 Task 1: Grade Prediction (Regression)
  Mean Absolute Error (MAE): 1.0258
  (On average, the model is off by 1.0258 points)

📋 Task 2: Romantic Status (Classification)
  Accuracy: 59.23%
  F1-Score (for 'yes' class): 0.1311
-------------------------------------------------------------


### Alpha Hyperparameter Tuning Results

| Alpha Value | Test MAE (Grade) | Test Accuracy % (Romantic) | Test F1-Score (Romantic) |
| :---: | :---: | :---: | :---: |
| 0.01 | 1.3284 | 49.23 | 0.3774 |
| 0.1 | 1.0683 | 56.92 | 0.4510 |
| 0.3 | 1.0418 | 66.92 | 0.2712 |
| 0.5 | 0.8784 | 60.77 | 0.0727 |
| 0.8 | 0.8693 | 63.85 | 0.1132 |